In [7]:
# 1. Put this repository first on sys.path, so `recipes` is its recipes/ package,
#    then construct the Cortex Training client from the connection file created during setup (~/cortex-training-config.json).
import contextlib
import io
import json
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "recipes" / "utils.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter inside a cortex-training checkout so `recipes` can be imported.")
sys.path.insert(0, str(REPO_ROOT))
import recipes  # noqa: E402

if Path(recipes.__file__).resolve().parent != REPO_ROOT / "recipes":
    raise RuntimeError(
        f"`recipes` resolves to {recipes.__file__}, outside {REPO_ROOT}. Another package named "
        "`recipes` is installed; uninstall it or restart the kernel, then run this cell again."
    )
from recipes.utils import make_config  # noqa: E402
from recipes.utils import make_client  # noqa: E402

import torch

from cortex_training import CortexTrainingClient, serialize_forward_backward_args

import os
config_path = Path(os.environ.get("CORTEX_TRAINING_CONFIG") or Path.home() / "cortex-training-config.json")
if not config_path.is_file():
    raise RuntimeError(
        f"Connection file not found at {config_path}. Create it as in docs/getting-started/setup.md "
        "(outside the repository), or set CORTEX_TRAINING_CONFIG to its path."
    )
# make_client reads the whole connection mapping -- host, PAT, database,
# schema, endpoint and options -- not just the token, so a configured file
# is honored in full.
client = make_client(str(config_path))

In [8]:
# 2. Build the Qwen3-8B SFT job from the model catalog. The override is for this demo
#    only: 128-token sequences in batches of 8 instead of the catalog's settings.
job_body = make_config(
    model="Qwen/Qwen3-8B",
    context="standard",
    overrides=[
        {
            "sub_job_configs": [
                {"job_type": "training", "training_config": {"max_seq_len": 128, "train_batch_size": 8}}
            ]
        }
    ],
)

In [9]:
# 3. Submit the job and wait until its workers are running (will take about 3 min).
job_id = client.create_job_from_body(job_body)["job_id"]
print(f"Job {job_id} is starting")
client.wait_for_job(job_id)
print(f"Job {job_id} is running")

Job 00000000-0000-0000-0000-000000000000 is starting
Job 00000000-0000-0000-0000-000000000000 is running


In [10]:
# 4. Generate five random SFT batches with causal next-token labels.
torch.manual_seed(42)
BATCH_SIZE = 8
SEQ_LEN = 128


def make_random_batch():
    input_ids = torch.randint(0, 1000, (BATCH_SIZE, SEQ_LEN))
    labels = torch.roll(input_ids, shifts=-1, dims=1)
    labels[:, -1] = -100
    return {
        "input_ids": input_ids,
        "position_ids": torch.arange(SEQ_LEN).expand(BATCH_SIZE, -1).contiguous(),
        "labels": labels,
    }


training_data = [make_random_batch() for _ in range(5)]

In [11]:
# 5. Run forward-backward and one optimizer update for each batch.
for step_number, batch in enumerate(training_data, start=1):
    payload = serialize_forward_backward_args(args=(), kwargs=batch)
    with contextlib.redirect_stdout(io.StringIO()):
        request_id = client.forward_backward(job_id, payload)
        result = client.poll_request(job_id, request_id)
        request_id = client.step(job_id, learning_rate=0.0001)
        client.poll_request(job_id, request_id)
    print(f"step={step_number} loss={result['avg_loss']}")

step=1 loss=10.504879236221313
step=2 loss=9.294854998588562
step=3 loss=8.82610309123993
step=4 loss=9.058582425117493
step=5 loss=8.27169132232666


In [ ]:
# 6. Tear down the remote job and release its GPUs.
client.cancel_job(job_id)
print(f"Cancelled job {job_id}")
job_id = None